In [1]:
import pandas as pd
import numpy as np
import datetime as dt

# 1. LOAD DATASET

In [2]:
file_path = r"C:/Users/SkyTech/Downloads/online_retail_II.csv/online_retail_II.csv"

print("Loading dataset...")
# Reading dataset (supports common encodings for Online Retail II)
try:
    df = pd.read_csv(file_path, encoding="ISO-8859-1")
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding="utf-8")

print(f"Raw shape: {df.shape}")

Loading dataset...
Raw shape: (541910, 8)


In [3]:
# Standardize column names (remove leading/trailing spaces)
df.columns = df.columns.str.strip()

In [4]:
# Renaming for consistency if needed:
# Typical Online Retail II columns: Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country
df.rename(columns = {'Customer ID':'CustomerID'}, inplace=True)

# 2. DATA CLEANING & TYPE CONVERSION


In [5]:
print("Cleaning data...")

# Drop rows missing CustomerID (cannot segment unknown customers)
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)

# Convert InvoiceDate to datetime (Fixed to avoid UserWarning)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed')

# Remove cancelled orders (Invoices starting with 'C') and negative/zero Quantities or Prices
df['Invoice'] = df['Invoice'].astype(str)
df = df[~df['Invoice'].str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Calculate Total spend per line item
df['TotalLineAmount'] = df['Quantity'] * df['Price']

print(f"Cleaned transactions shape: {df.shape}")

# Save cleaned transactional dataset for SQL / Power BI
df.to_csv('cleaned_online_retail.csv', index=False)
print("Saved 'cleaned_online_retail.csv'")

Cleaning data...
Cleaned transactions shape: (397885, 9)
Saved 'cleaned_online_retail.csv'


# 3. BUILD RFM METRICS (Recency, Frequency, Monetary)


In [6]:
print("Calculating RFM metrics...")

# Set reference analysis date (1 day after the latest transaction in dataset)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency: Days since last order
    'Invoice': 'nunique',                                   # Frequency: Total unique orders
    'TotalLineAmount': 'sum'                                 # Monetary: Total spend
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Round Monetary values to 2 decimal places
rfm['Monetary'] = rfm['Monetary'].round(2)

Calculating RFM metrics...


# 4. RFM SCORING (1 to 5 Quantiles)


In [7]:
# Recency: Lower days = better score (1 = inactive long ago, 5 = bought recently)
# Frequency & Monetary: Higher value = better score (1 = lowest, 5 = highest)

rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

# Convert scores to integers for conditional logic
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)

# Combine scores into RFM Cell Code (e.g., '555')
rfm['RFM_Cell'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# 5. CUSTOMER SEGMENTATION LOGIC


In [8]:
def segment_customer(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f <= 2:
        return 'Recent / Promising'
    elif r == 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r == 1 and f >= 4 and m >= 4:
        return 'Cant Lose Them'
    elif r <= 2 and f <= 2:
        return 'Hibernating / Lost'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# 7. SUMMARY & EXPORT


In [9]:
print("\n--- RFM SEGMENTATION SUMMARY ---")
summary = rfm.groupby('Segment').agg(
    Customer_Count=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).reset_index()

print(summary)


--- RFM SEGMENTATION SUMMARY ---
              Segment  Customer_Count  Avg_Recency  Avg_Frequency  \
0             At Risk             343   111.766764       3.889213   
1      Cant Lose Them              25   232.320000       6.360000   
2           Champions             962    12.861746      11.080042   
3  Hibernating / Lost            1065   217.897653       1.101408   
4     Loyal Customers             998    34.091182       3.714429   
5     Needs Attention             275   196.854545       2.530909   
6  Recent / Promising             670    36.846269       1.200000   

   Total_Revenue  
0      526713.40  
1       53066.52  
2     5809341.07  
3      519408.57  
4     1474145.55  
5      220751.63  
6      307999.16  


In [10]:
# 1. Create the 'Is_At_Risk' flag based on your Segment values
# Adjust the segment names in the list to match your exact RFM naming scheme
at_risk_segments = ['At Risk', 'At-Risk', 'Cant Lose Them', 'About to Sleep']

rfm['Is_At_Risk'] = rfm['Segment'].isin(at_risk_segments).astype(int)

# 2. Calculate total revenue at risk
total_revenue = rfm['Monetary'].sum()
at_risk_revenue = rfm[rfm['Is_At_Risk'] == 1]['Monetary'].sum()
at_risk_pct = (at_risk_revenue / total_revenue) * 100

print(f"\nTotal Portfolio Revenue: ${total_revenue:,.2f}")
print(f"Total Revenue At Risk: ${at_risk_revenue:,.2f} ({at_risk_pct:.2f}%)")



Total Portfolio Revenue: $8,911,425.90
Total Revenue At Risk: $579,779.92 (6.51%)


In [11]:
print(rfm.columns)

Index(['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score',
       'M_Score', 'RFM_Cell', 'Segment', 'Is_At_Risk'],
      dtype='str')


In [12]:
# Export scored dataset
rfm.to_csv("rfm_customer_segments.csv", index=False)
print("\nExported 'rfm_customer_segments.csv' successfully!")


Exported 'rfm_customer_segments.csv' successfully!
